# Final 4B — recipe A+B

attribute natural; title/description cân bằng theo B. 8M token / 1.000 step.


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('no GPU')
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9, 1), 'GB')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)


In [ ]:
DATA = Path('/kaggle/input/tiki-weight-final-data')
if not (DATA / 'units_F.parquet').exists():
    hits = sorted(Path('/kaggle/input').rglob('units_F.parquet'))
    assert hits, 'không thấy units_F.parquet'
    DATA = hits[0].parent
ROOT = Path('/kaggle/working/f_data')
(ROOT / 'processed_F').mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA / 'units_F.parquet', ROOT / 'processed_F/units.parquet')
shutil.copy2(DATA / 'eval_F.parquet', ROOT / 'eval_F.parquet')
shutil.copy2(DATA / 'meta_F.json', ROOT / 'meta_F.json')

CODE = Path('/kaggle/working/code'); CODE.mkdir(exist_ok=True)
trainer = sorted(Path('/kaggle/input').rglob('train_weight_quota_gpu.py'))[0]
for name in ('train_weight_quota_gpu.py', 'quota_sampling.py', 'tpu_weight_train.py'):
    shutil.copy2(trainer.parent / name, CODE / name)
tr = CODE / 'train_weight_quota_gpu.py'
tr.write_text(tr.read_text().replace('choices=("A", "B")', 'choices=("A", "B", "C", "F")'), encoding='utf-8')

MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try:
        d = json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        continue
    if (d.get('hidden_size'), d.get('intermediate_size'), d.get('num_hidden_layers')) == (2560, 9728, 36):
        MODEL = str(cfg.parent); break
assert MODEL, 'không thấy Qwen3-4B'
print('DATA', DATA, '| MODEL', MODEL)


In [ ]:
cmd = [sys.executable, str(CODE / 'train_weight_quota_gpu.py'),
       '--dataset', 'F', '--data-root', str(ROOT),
       '--out', '/kaggle/working/final_4b_4m', '--model', MODEL,
       '--seed', '42', '--tokens', '4000000', '--steps', '500',
       '--configs', 'final']
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
out=Path('/kaggle/working/final_4b_4m')
for p in sorted(out.glob('hyp_*.parquet')): print(p.name, p.stat().st_size)
print((out/'summary.json').read_text())
